In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('online_retail_II.csv')

In [3]:
data.shape

(1048575, 8)

In [4]:
data = data[data['Quantity'] > 0]
data = data[data['Price'] > 0]
data = data.dropna(subset=['Description'])
data=data[data['Country']=='United Kingdom']

In [5]:
data['Revenue'] = data['Quantity'] * data['Price']
data.shape

(936056, 9)

In [6]:
data.to_csv('retail_clean.csv', index=False)

In [7]:
print(data.shape)
print(data['Description'].nunique(), "unique products")

(936056, 9)
5333 unique products


In [8]:
def assign_category(description):
    desc = str(description).upper()
    
    if any(k in desc for k in ['CHRISTMAS', 'XMAS', 'SANTA', 'REINDEER', 'BAUBLE', 'TINSEL', 'ADVENT']):
        return 'Seasonal & Christmas'
    elif any(k in desc for k in ['MUG', 'CUP', 'BOWL', 'PLATE', 'SPOON', 'FORK', 'KNIFE', 
                                   'CUTLERY', 'LUNCHBOX', 'CAKE', 'TEAPOT', 'KITCHEN', 'BAKING',
                                   'RECIPE', 'COOK', 'APRON', 'OVEN', 'PANTRY']):
        return 'Kitchen & Dining'
    elif any(k in desc for k in ['BAG', 'TOTE', 'SHOPPER', 'PURSE', 'WALLET', 'BACKPACK', 
                                   'SATCHEL', 'HANDBAG', 'POUCH', 'CLUTCH']):
        return 'Bags & Accessories'
    elif any(k in desc for k in ['FRAME', 'SIGN', 'PLAQUE', 'BANNER', 'BUNTING', 'POSTER',
                                   'NOTEBOOK', 'PEN', 'PENCIL', 'RULER', 'DIARY', 'JOURNAL',
                                   'LETTER', 'STAMP', 'CHALK', 'BLACKBOARD']):
        return 'Stationery & Signage'
    elif any(k in desc for k in ['CANDLE', 'HOLDER', 'LANTERN', 'LIGHT', 'LAMP', 'FAIRY',
                                   'TEALIGHT', 'WICKER', 'SCENT', 'DIFFUSER', 'INCENSE']):
        return 'Lighting & Candles'
    elif any(k in desc for k in ['BOX', 'TIN', 'BASKET', 'STORAGE', 'RACK', 'SHELF',
                                   'DRAWER', 'CABINET', 'ORGANISER', 'CONTAINER', 'JAR', 'CRATE']):
        return 'Storage & Organisation'
    elif any(k in desc for k in ['HEART', 'ROSE', 'FLOWER', 'FLORAL', 'GARDEN', 'PLANT', 
                                   'HERB', 'SEED', 'BOTANICAL', 'BUTTERFLY', 'BEE', 'BIRD',
                                   'NATURE', 'LEAF', 'TREE', 'WOODLAND']):
        return 'Garden & Floral'
    elif any(k in desc for k in ['CARD', 'WRAP', 'RIBBON', 'TAG', 'STICKER', 'GIFT',
                                   'WRAP', 'BOW', 'TISSUE', 'CELLOPHANE', 'ENVELOPE']):
        return 'Gifting & Wrapping'
    elif any(k in desc for k in ['CLOCK', 'MIRROR', 'CUSHION', 'DOORMAT', 'CURTAIN', 'RUG',
                                   'WALL', 'DECOR', 'ORNAMENT', 'FIGURINE', 'STATUE', 'PRINT',
                                   'PHOTO', 'VINTAGE', 'RETRO', 'SHABBY', 'CHIC', 'ANTIQUE']):
        return 'Home Decor'
    elif any(k in desc for k in ['TOY', 'GAME', 'PUZZLE', 'DOLL', 'CHILDREN', 'KIDS', 'BABY',
                                   'PLAY', 'FUN', 'ANIMAL', 'DINOSAUR', 'TEDDY', 'BEAR', 'FAIRY']):
        return 'Toys & Games'
    elif any(k in desc for k in ['SOAP', 'BATH', 'SHOWER', 'TOWEL', 'SPONGE', 'BRUSH',
                                   'BEAUTY', 'CREAM', 'LOTION', 'PERFUME', 'COLOGNE', 'SCENTED']):
        return 'Health & Beauty'
    elif any(k in desc for k in ['SHIRT', 'DRESS', 'HAT', 'SCARF', 'GLOVE', 'SOCK',
                                   'APRON', 'UMBRELLA', 'FASHION', 'WEAR', 'CLOTH']):
        return 'Clothing & Accessories'
    else:
        return 'General & Other'

data['Category'] = data['Description'].apply(assign_category)

In [9]:
print(data[data['Category'] == 'General & Other']['Description'].count() / len(data) * 100)

13.689992906407308


In [10]:
margin_map = {
    'Seasonal & Christmas':     0.55,
    'Kitchen & Dining':         0.45,
    'Bags & Accessories':       0.40,
    'Stationery & Signage':     0.35,
    'Lighting & Candles':       0.50,
    'Storage & Organisation':   0.50,
    'Garden & Floral':          0.45,
    'Gifting & Wrapping':       0.40,
    'Home Decor':               0.40,
    'Toys & Games':             0.50,
    'Health & Beauty':          0.35,
    'Clothing & Accessories':   0.45,
    'General & Other':          0.50,
}

# Fix: map first
data['Cost_Pct']      = data['Category'].map(margin_map)
data['Cost_Price']    = data['Price'] * data['Cost_Pct']
data['Revenue']       = data['Quantity'] * data['Price']
data['Total_Cost']    = data['Quantity'] * data['Cost_Price']
data['Gross_Profit']  = data['Revenue'] - data['Total_Cost']
data['Margin_Pct']    = (data['Gross_Profit'] / data['Revenue'] * 100).round(2)

In [31]:
# --- Summary by category ---
summary = data.groupby('Category').agg(
    Total_Revenue   = ('Revenue', 'sum'),
    Total_Cost      = ('Total_Cost', 'sum'),
    Gross_Profit    = ('Gross_Profit', 'sum'),
    Avg_Margin_Pct  = ('Margin_Pct', 'mean'),
    Transactions    = ('Invoice', 'nunique'),
    Unique_Products = ('StockCode', 'nunique')
).reset_index()

summary['Avg_Margin_Pct'] = summary['Avg_Margin_Pct'].round(2)
summary = summary.sort_values('Gross_Profit', ascending=False)

print(summary.to_string())

data.to_csv('retail_analysed.csv', index=False)
summary.to_csv('category_summary.csv', index=False)

                  Category  Total_Revenue    Total_Cost  Gross_Profit  Avg_Margin_Pct  Transactions  Unique_Products
3          General & Other    9266578.022  4.633289e+06  4.633289e+06            50.0         26339             1055
4       Gifting & Wrapping    5665020.070  2.266008e+06  3.399012e+06            60.0         13982              295
7         Kitchen & Dining    5287932.240  2.379570e+06  2.908363e+06            55.0         25575              575
10    Stationery & Signage    3792723.820  1.327453e+06  2.465270e+06            65.0         22746              440
6               Home Decor    3566408.337  1.426563e+06  2.139845e+06            60.0         19293              355
11  Storage & Organisation    3948172.280  1.974086e+06  1.974086e+06            50.0         22381              415
2          Garden & Floral    3041609.450  1.368724e+06  1.672885e+06            55.0         22074              623
8       Lighting & Candles    3316924.850  1.658462e+06  1.65846

In [12]:
print(data.head())

      Invoice StockCode                        Description  Quantity  \
22697  491155     82482  WOODEN PICTURE FRAME WHITE FINISH         1   
22698  491155     82551             LAUNDRY 15C METAL SIGN         1   
22699  491155     82553      POP'S GREASY SPOON METAL SIGN         1   
22700  491155     82567          AIRLINE LOUNGE,METAL SIGN         1   
22701  491155     82578                 KITCHEN METAL SIGN         1   

            InvoiceDate  Price  Customer ID         Country  Revenue  \
22697  10-12-2009 09:49   5.17          NaN  United Kingdom     5.17   
22698  10-12-2009 09:49   2.57          NaN  United Kingdom     2.57   
22699  10-12-2009 09:49   3.43          NaN  United Kingdom     3.43   
22700  10-12-2009 09:49   4.30          NaN  United Kingdom     4.30   
22701  10-12-2009 09:49   1.30          NaN  United Kingdom     1.30   

                   Category  Cost_Pct  Cost_Price  Total_Cost  Gross_Profit  \
22697  Stationery & Signage      0.35      1.8095      

In [13]:
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], dayfirst=True)
data['YearMonth'] = data['InvoiceDate'].dt.to_period('M').astype(str)

In [14]:
monthly = data.groupby(['YearMonth', 'Category']).agg(
    Revenue = ('Revenue', 'sum'),
    Gross_Profit = ('Gross_Profit', 'sum')
).reset_index()

monthly['Margin_Pct'] = ((monthly['Gross_Profit']/monthly['Revenue'])*100).round(2)


In [15]:
monthly.head()

,YearMonth,Category,Revenue,Gross_Profit,Margin_Pct
0,2009-12,Bags & Accessories,3689.47,2213.6820,60.0
1,2009-12,Clothing & Accessories,1744.53,959.4915,55.0
2,2009-12,Garden & Floral,8621.71,4741.9405,55.0
3,2009-12,General & Other,21972.53,10986.2650,50.0
4,2009-12,Gifting & Wrapping,12657.34,7594.4040,60.0


In [24]:
product_summary = data.groupby(['StockCode', 'Description', 'Category']).agg(
    Revenue = ('Revenue', 'sum'),
    Gross_Profit = ('Gross_Profit', 'sum'),
    Units_Sold = ('Quantity', 'sum'),
    Avg_Price = ('Price', 'mean')
).reset_index()

In [25]:
product_summary['Margin_Pct'] = ((product_summary['Gross_Profit']/product_summary['Revenue'])*100).round(2)

In [27]:
# Top 20 products
top_products = product_summary.sort_values('Gross_Profit', ascending=False).head(20)

In [28]:
# Underperformers = low volume AND low revenue despite being in the catalogue
underperformers = product_summary[product_summary['Units_Sold'] > 100].copy()

# Flag products earning low revenue relative to their category peers
category_avg = product_summary.groupby('Category')['Revenue'].transform('mean')
product_summary['Revenue_vs_Category_Avg'] = (product_summary['Revenue'] / category_avg * 100).round(2)

underperformers = product_summary[
    (product_summary['Units_Sold'] > 100) &
    (product_summary['Revenue_vs_Category_Avg'] < 50)  # earning less than 50% of category average
].sort_values('Revenue_vs_Category_Avg').head(20)

In [32]:
summary = pd.read_csv('category_summary.csv')

def flag(row):
    if row['Avg_Margin_Pct'] >= 60 and row['Total_Revenue'] > 500000:
        return 'Star — Protect & Grow'
    elif row['Avg_Margin_Pct'] < 50:
        return 'Risk — Review Pricing'
    elif row['Unique_Products'] > 400 and row['Avg_Margin_Pct'] < 55:
        return 'Complex — Rationalise Range'
    elif row['Total_Revenue'] < 300000 and row['Avg_Margin_Pct'] >= 55:
        return 'Hidden Gem — Expand'
    else:
        return 'Stable — Monitor'

summary['Commercial_Flag'] = summary.apply(flag, axis=1)

# --- Export to Excel with multiple sheets ---
with pd.ExcelWriter('retail_commercial_analysis.xlsx', engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='Category Summary', index=False)
    monthly.to_excel(writer, sheet_name='Monthly Trends', index=False)
    top_products.to_excel(writer, sheet_name='Top 20 Products', index=False)
    underperformers.to_excel(writer, sheet_name='Underperformers', index=False)

print("Excel file saved: retail_commercial_analysis.xlsx")

Excel file saved: retail_commercial_analysis.xlsx
